# CSR attention CUDA JIT/reference check

This notebook JIT-compiles the CSR attention `.cu` file with a temporary pybind binding, defines a temporary Python autograd wrapper, builds a bounded-degree random CSR graph with about 200k edges, and compares forward/backward against a native PyTorch segment-softmax reference.

In [1]:
from pathlib import Path
import os
import tempfile
import time

import pandas as pd
import torch
from torch.utils.cpp_extension import load

ROOT = Path('/mnt/nvmefs/Projects/SymTRELLIS')
CUDA_SRC = ROOT / 'symtrellis/mapper/attention/csr_attn_ext/csr_attn_cuda.cu'
CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', '/home/quanta/.conda/envs/symm-enforce'))

os.environ['PATH'] = f'{CONDA_PREFIX / "bin"}:{os.environ.get("PATH", "")}'
os.environ['CC'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['CXX'] = str(CONDA_PREFIX / 'bin/g++')
os.environ['CUDAHOSTCXX'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'
assert CUDA_SRC.exists(), CUDA_SRC

if 'TORCH_CUDA_ARCH_LIST' not in os.environ:
    major, minor = torch.cuda.get_device_capability(DEVICE)
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'

torch.manual_seed(20260628)
torch.cuda.manual_seed_all(20260628)

print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('CUDA_SRC:', CUDA_SRC)
print('TORCH_CUDA_ARCH_LIST:', os.environ['TORCH_CUDA_ARCH_LIST'])
print('MAX_JOBS:', os.environ['MAX_JOBS'])

torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
CUDA_SRC: /mnt/nvmefs/Projects/SymTRELLIS/symtrellis/mapper/attention/csr_attn_ext/csr_attn_cuda.cu
TORCH_CUDA_ARCH_LIST: 8.9
MAX_JOBS: 32


## Temporary JIT binding

The binding is written into a temporary directory. It declares the two C++ entry points implemented by `csr_attn_cuda.cu` and exposes them under the same names expected by the Python wrapper below.

In [2]:
jit_root = Path(tempfile.mkdtemp(prefix='csr_attn_jit_'))
src_dir = jit_root / 'src'
build_dir = jit_root / 'torch_ext'
src_dir.mkdir(parents=True, exist_ok=True)
build_dir.mkdir(parents=True, exist_ok=True)

binding_cpp = src_dir / 'binding.cpp'
binding_cpp.write_text(r'''
#include <torch/extension.h>
#include <vector>

std::vector<torch::Tensor> sparse_csr_attn_forward(
    torch::Tensor q,
    torch::Tensor k,
    torch::Tensor v,
    torch::Tensor rowptr,
    torch::Tensor col);

std::vector<torch::Tensor> sparse_csr_attn_backward(
    torch::Tensor grad_out,
    torch::Tensor q,
    torch::Tensor k,
    torch::Tensor v,
    torch::Tensor rowptr,
    torch::Tensor col,
    torch::Tensor out,
    torch::Tensor lse);

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("sparse_csr_attn_forward", &sparse_csr_attn_forward);
  m.def("sparse_csr_attn_backward", &sparse_csr_attn_backward);
}
''')

ext_name = f'csr_attn_jit_{os.getpid()}_{int(time.time())}'
ext = load(
    name=ext_name,
    sources=[str(binding_cpp), str(CUDA_SRC)],
    build_directory=str(build_dir),
    with_cuda=True,
    extra_cuda_cflags=['-O3'],
    verbose=True,
)
print('loaded:', ext_name)
print('jit_root:', jit_root)

Detected CUDA files, patching ldflags
Emitting ninja build file /tmp/csr_attn_jit_1cwgcl93/torch_ext/build.ninja...
Building extension module csr_attn_jit_648864_1782701801...
Using envvar MAX_JOBS (32) as the number of workers...


[1/3] /home/quanta/.conda/envs/symm-enforce/bin/g++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=csr_attn_jit_648864_1782701801 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /tmp/csr_attn_jit_1cwgcl93/src/binding.cpp -o binding.o 
[2/3] /home/quanta/.conda/envs/symm-enforce/bin/nvcc --generate-dependencies-with-compile --dependency-output csr_attn_cud

Loading extension module csr_attn_jit_648864_1782701801...


## Temporary Python wrapper

In [7]:
class SparseCSRAttentionFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, rowptr, col):
        out, lse = ext.sparse_csr_attn_forward(q, k, v, rowptr, col)
        ctx.save_for_backward(q, k, v, rowptr, col, out, lse)
        return out

    @staticmethod
    def backward(ctx, grad_out):
        q, k, v, rowptr, col, out, lse = ctx.saved_tensors
        dq, dk, dv = ext.sparse_csr_attn_backward(
            grad_out.contiguous(), q, k, v, rowptr, col, out, lse
        )
        return dq, dk, dv, None, None


def csr_attn_cuda(q, k, v, rowptr, col):
    assert q.is_cuda and k.is_cuda and v.is_cuda
    assert rowptr.is_cuda and col.is_cuda
    assert q.dtype in (torch.float32, torch.float16, torch.bfloat16)
    assert q.dtype == k.dtype == v.dtype
    assert q.ndim == k.ndim == v.ndim == 3
    assert k.shape == v.shape
    assert q.shape[1:] == k.shape[1:]
    assert q.shape[-1] in (16, 32, 64, 128)
    assert rowptr.dtype == torch.int32 and col.dtype == torch.int32
    assert rowptr.ndim == 1 and rowptr.numel() == q.shape[0] + 1
    assert col.ndim == 1
    return SparseCSRAttentionFn.apply(
        q.contiguous(), k.contiguous(), v.contiguous(), rowptr.contiguous(), col.contiguous()
    )

## Native PyTorch reference and bounded-degree graph

The reference materializes edge-level scores/probabilities and uses segment reductions. It is intentionally only a correctness reference, not the memory-efficient implementation.

In [8]:
def csr_attn_reference(q, k, v, rowptr, col):
    Nq, H, D = q.shape
    E = col.numel()
    col64 = col.to(torch.long)
    deg = (rowptr[1:] - rowptr[:-1]).to(torch.long)
    rows = torch.repeat_interleave(
        torch.arange(Nq, device=q.device, dtype=torch.long), deg, output_size=E
    )

    heads = []
    scale = D ** -0.5
    for h in range(H):
        scores = (q[rows, h].float() * k[col64, h].float()).sum(dim=-1) * scale
        with torch.no_grad():
            row_max = torch.full((Nq,), -torch.inf, device=q.device, dtype=torch.float32)
            row_max.scatter_reduce_(0, rows, scores.detach(), reduce='amax', include_self=True)

        weights = torch.exp(scores - row_max[rows])
        row_sum = torch.zeros((Nq,), device=q.device, dtype=torch.float32)
        row_sum.index_add_(0, rows, weights)
        probs = weights / row_sum[rows]

        out_h = torch.zeros((Nq, D), device=q.device, dtype=torch.float32)
        out_h.index_add_(0, rows, probs[:, None] * v[col64, h].float())
        heads.append(out_h)

    return torch.stack(heads, dim=1).to(q.dtype)


def make_bounded_degree_csr(device):
    Nq = 20_000
    Nk = 24_000
    H = 4
    D = 32
    degrees = torch.full((Nq,), 10, device=device, dtype=torch.int32)
    degrees[0::20] = 0
    degrees[1::20] = 16
    degrees[2::20] = 14

    rowptr = torch.empty((Nq + 1,), device=device, dtype=torch.int32)
    rowptr[0] = 0
    rowptr[1:] = torch.cumsum(degrees, dim=0, dtype=torch.int32)
    E = int(rowptr[-1].item())
    col = torch.randint(0, Nk, (E,), device=device, dtype=torch.int32)
    return rowptr.contiguous(), col.contiguous(), degrees, Nq, Nk, H, D


rowptr, col, degrees, Nq, Nk, H, D = make_bounded_degree_csr(DEVICE)
print('Nq:', Nq, 'Nk:', Nk, 'H:', H, 'D:', D, 'E:', col.numel())
print('degree min/mean/max:', int(degrees.min().item()), float(degrees.float().mean().item()), int(degrees.max().item()))
assert col.numel() == 200_000
assert int(degrees.max().item()) == 16

Nq: 20000 Nk: 24000 H: 4 D: 32 E: 200000
degree min/mean/max: 0 10.0 16


## Forward/backward equivalence

In [9]:
dtype = torch.float32
q = (torch.randn((Nq, H, D), device=DEVICE, dtype=dtype) * 0.2).requires_grad_(True)
k = (torch.randn((Nk, H, D), device=DEVICE, dtype=dtype) * 0.2).requires_grad_(True)
v = torch.randn((Nk, H, D), device=DEVICE, dtype=dtype).requires_grad_(True)

q_ref = q.detach().clone().requires_grad_(True)
k_ref = k.detach().clone().requires_grad_(True)
v_ref = v.detach().clone().requires_grad_(True)

torch.cuda.synchronize()
out_cuda = csr_attn_cuda(q, k, v, rowptr, col)
out_ref = csr_attn_reference(q_ref, k_ref, v_ref, rowptr, col)

grad_out = torch.randn_like(out_cuda)
(out_cuda * grad_out).sum().backward()
(out_ref * grad_out).sum().backward()
torch.cuda.synchronize()


def compare(name, actual, expected, atol, rtol):
    diff = (actual - expected).detach().abs()
    max_abs = float(diff.max().item())
    mean_abs = float(diff.mean().item())
    passed = bool(torch.allclose(actual, expected, atol=atol, rtol=rtol))
    return {
        'name': name,
        'passed': passed,
        'max_abs': max_abs,
        'mean_abs': mean_abs,
        'atol': atol,
        'rtol': rtol,
    }


rows = [
    compare('out', out_cuda, out_ref, atol=5e-5, rtol=5e-4),
    compare('dq', q.grad, q_ref.grad, atol=2e-4, rtol=2e-3),
    compare('dk', k.grad, k_ref.grad, atol=5e-4, rtol=5e-3),
    compare('dv', v.grad, v_ref.grad, atol=5e-4, rtol=5e-3),
]
df = pd.DataFrame(rows)
display(df)
assert df['passed'].all(), df

,name,passed,max_abs,mean_abs,atol,rtol
0,out,True,3.576279e-07,2.124409e-08,0.00005,0.0005
1,dq,True,1.788139e-07,7.772222e-09,0.00020,0.0020
2,dk,True,1.192093e-07,7.160626e-09,0.00050,0.0050
3,dv,True,4.768372e-07,3.082289e-08,0.00050,0.0050


In [10]:
# Dtype correctness plus repeated forward+backward runtime/memory comparison.
BENCH_REPEATS = 5
DTYPE_CASES = [
    ('float32', torch.float32, {'out': (5e-5, 5e-4), 'dq': (2e-4, 2e-3), 'dk': (5e-4, 5e-3), 'dv': (5e-4, 5e-3)}),
    ('float16', torch.float16, {'out': (5e-3, 5e-2), 'dq': (2e-2, 5e-2), 'dk': (2e-2, 5e-2), 'dv': (2e-2, 5e-2)}),
    ('bfloat16', torch.bfloat16, {'out': (4e-2, 8e-2), 'dq': (6e-2, 1e-1), 'dk': (6e-2, 1e-1), 'dv': (6e-2, 1e-1)}),
]


def make_qkv(dtype, seed):
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(seed)
    q = (torch.randn((Nq, H, D), device=DEVICE, dtype=dtype, generator=gen) * 0.2).requires_grad_(True)
    k = (torch.randn((Nk, H, D), device=DEVICE, dtype=dtype, generator=gen) * 0.2).requires_grad_(True)
    v = torch.randn((Nk, H, D), device=DEVICE, dtype=dtype, generator=gen).requires_grad_(True)
    grad_out = torch.randn((Nq, H, D), device=DEVICE, dtype=dtype, generator=gen)
    return q, k, v, grad_out


def dtype_correctness(dtype_name, dtype, tolerances):
    q, k, v, grad_out = make_qkv(dtype, seed=20260628)
    q_ref = q.detach().clone().requires_grad_(True)
    k_ref = k.detach().clone().requires_grad_(True)
    v_ref = v.detach().clone().requires_grad_(True)

    out_cuda = csr_attn_cuda(q, k, v, rowptr, col)
    out_ref = csr_attn_reference(q_ref, k_ref, v_ref, rowptr, col)
    (out_cuda * grad_out).sum().backward()
    (out_ref * grad_out).sum().backward()
    torch.cuda.synchronize()

    items = [('out', out_cuda, out_ref), ('dq', q.grad, q_ref.grad), ('dk', k.grad, k_ref.grad), ('dv', v.grad, v_ref.grad)]
    rows = []
    for name, actual, expected in items:
        atol, rtol = tolerances[name]
        diff = (actual.float() - expected.float()).detach().abs()
        rows.append({
            'dtype': dtype_name,
            'name': name,
            'passed': bool(torch.allclose(actual.float(), expected.float(), atol=atol, rtol=rtol)),
            'max_abs': float(diff.max().item()),
            'mean_abs': float(diff.mean().item()),
            'atol': atol,
            'rtol': rtol,
        })
    del q, k, v, q_ref, k_ref, v_ref, grad_out, out_cuda, out_ref
    torch.cuda.synchronize()
    return rows


def timed_fwd_bwd(method_name, method, dtype, seed):
    q_base, k_base, v_base, grad_out = make_qkv(dtype, seed=seed)
    q_base = q_base.detach(); k_base = k_base.detach(); v_base = v_base.detach(); grad_out = grad_out.detach()

    def one_run(measured):
        q = q_base.clone().requires_grad_(True)
        k = k_base.clone().requires_grad_(True)
        v = v_base.clone().requires_grad_(True)
        torch.cuda.synchronize()
        if measured:
            torch.cuda.reset_peak_memory_stats()
        base_alloc = torch.cuda.memory_allocated()
        base_reserved = torch.cuda.memory_reserved()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        out = method(q, k, v)
        loss = (out * grad_out).sum()
        loss.backward()
        end.record()
        torch.cuda.synchronize()
        record = {
            'ms': float(start.elapsed_time(end)),
            'alloc_delta_mb': (torch.cuda.memory_allocated() - base_alloc) / 1024**2,
            'reserved_delta_mb': (torch.cuda.memory_reserved() - base_reserved) / 1024**2,
            'peak_alloc_delta_mb': (torch.cuda.max_memory_allocated() - base_alloc) / 1024**2,
            'peak_reserved_delta_mb': (torch.cuda.max_memory_reserved() - base_reserved) / 1024**2,
        }
        del q, k, v, out, loss
        torch.cuda.synchronize()
        return record

    _ = one_run(measured=False)
    records = [one_run(measured=True) for _ in range(BENCH_REPEATS)]
    del q_base, k_base, v_base, grad_out
    torch.cuda.synchronize()
    return {
        'method': method_name,
        'runtime_ms_mean': sum(r['ms'] for r in records) / len(records),
        'runtime_ms_min': min(r['ms'] for r in records),
        'alloc_delta_mb_mean': sum(r['alloc_delta_mb'] for r in records) / len(records),
        'reserved_delta_mb_mean': sum(r['reserved_delta_mb'] for r in records) / len(records),
        'peak_alloc_delta_mb_mean': sum(r['peak_alloc_delta_mb'] for r in records) / len(records),
        'peak_reserved_delta_mb_mean': sum(r['peak_reserved_delta_mb'] for r in records) / len(records),
    }


correctness_rows = []
bench_rows = []
for dtype_name, dtype, tolerances in DTYPE_CASES:
    print(f'=== dtype {dtype_name} correctness ===')
    correctness_rows.extend(dtype_correctness(dtype_name, dtype, tolerances))
    methods = [
        ('cuda', lambda q, k, v: csr_attn_cuda(q, k, v, rowptr, col)),
        ('reference', lambda q, k, v: csr_attn_reference(q, k, v, rowptr, col)),
    ]
    for method_name, method in methods:
        print('bench', dtype_name, method_name)
        row = timed_fwd_bwd(method_name, method, dtype, seed=20260629)
        row['dtype'] = dtype_name
        row['repeats'] = BENCH_REPEATS
        bench_rows.append(row)

correctness_by_dtype = pd.DataFrame(correctness_rows)
bench_by_dtype = pd.DataFrame(bench_rows)
display(correctness_by_dtype)
display(bench_by_dtype[[
    'dtype', 'method', 'repeats', 'runtime_ms_mean', 'runtime_ms_min',
    'alloc_delta_mb_mean', 'reserved_delta_mb_mean',
    'peak_alloc_delta_mb_mean', 'peak_reserved_delta_mb_mean',
]])
assert correctness_by_dtype['passed'].all(), correctness_by_dtype

=== dtype float32 correctness ===
bench float32 cuda
bench float32 reference
=== dtype float16 correctness ===
bench float16 cuda
bench float16 reference
=== dtype bfloat16 correctness ===
bench bfloat16 cuda
bench bfloat16 reference


,dtype,name,passed,max_abs,mean_abs,atol,rtol
0,float32,out,True,3.576279e-07,2.125692e-08,0.00005,0.0005
1,float32,dq,True,1.490116e-07,7.799722e-09,0.00020,0.0020
2,float32,dk,True,1.192093e-07,7.172805e-09,0.00050,0.0050
3,float32,dv,True,4.172325e-07,3.088016e-08,0.00050,0.0050
4,float16,out,True,4.882812e-04,2.119138e-08,0.00500,0.0500
5,float16,dq,True,2.441406e-04,9.166234e-06,0.02000,0.0500
6,float16,dk,True,7.324219e-04,1.584216e-05,0.02000,0.0500
7,float16,dv,True,1.953125e-03,8.006526e-05,0.02000,0.0500
8,bfloat16,out,True,3.906250e-03,2.033600e-08,0.04000,0.0800
9,bfloat16,dq,True,1.953125e-03,7.330278e-05,0.06000,0.1000


,dtype,method,repeats,runtime_ms_mean,runtime_ms_min,alloc_delta_mb_mean,reserved_delta_mb_mean,peak_alloc_delta_mb_mean,peak_reserved_delta_mb_mean
0,float32,cuda,5,0.577069,0.504704,43.437988,0.0,53.509277,0.0
1,float32,reference,5,5.737427,5.497856,43.437988,0.0,473.135742,0.0
2,float16,cuda,5,0.765542,0.720896,22.094238,0.0,27.282715,0.0
3,float16,reference,5,5.522387,5.253120,23.051270,0.0,469.637207,0.0
4,bfloat16,cuda,5,0.773446,0.705792,22.094238,0.0,27.282715,0.0
5,bfloat16,reference,5,6.318918,6.095872,23.051270,0.0,469.637207,0.0
